In [ ]:
from option_analyzer import *
self = OptionAnalyzer('quotes', 'chain')

In [ ]:
option_type = 'call'
servers = sorted(set([f.split('~')[1] for f in glob(os.path.expanduser(f'~/lab/data/{option_type}~*~*.csv'))]))
latest_option_files = [sorted(glob(os.path.expanduser(f'~/lab/data/{option_type}~{svr}~*.csv')))[-1] for svr in servers]
print('\n'.join(map(os.path.basename, latest_option_files)))
chain_file_mtimes = dict([(os.path.basename(_f), os.path.getmtime(_f)) for _f in glob(os.path.expanduser('~/lab/chain/*'))])
latest_symbol = sorted(chain_file_mtimes, key=chain_file_mtimes.get)[-1]
print(latest_symbol, datetime.fromtimestamp(chain_file_mtimes[latest_symbol]).strftime('%F %T'))

In [ ]:
dfc = pd.concat([pd.read_csv(_f) for _f in latest_option_files])

### Call Options: ignore no-bid or low open interest (minimum open interests is 100)
Buy calls to maximize leverage

In [ ]:
dte_lb = 90
hdte_resid_lb = 0.95
overpaid_ub = 0.1
spread_ub = 15
_filter = (dfc.dte >= dte_lb) & (dfc.hdte_resid >= hdte_resid_lb) & (dfc.overpaid <= overpaid_ub) & (dfc.pctSpread <= spread_ub)
_filter = _filter  & (dfc.moneyness <= 1.0) & (dfc.symbol != 'TLT') #& (dfc.symbol != 'SPY')
_dfc = dfc[_filter].drop(columns=['dthr', 'dtzr']).sort_values(by='leverage', ascending=False)
_dfc.head(20)

### Top leverage

In [ ]:
_filter = (dfc.pctSpread <= 5) & (dfc.moneyness <= 1) & (dfc.dte >= 60) & (dfc.symbol != 'TLT')
px.scatter(dfc[_filter].sort_values(by='leverage', ascending=False).head(1000), x='hdte_resid', y='leverage', color='symbol', height=600)

In [ ]:
px.scatter(dfc[(dfc.symbol=='GOOGL') & (dfc.expDt == '2026-06-18') & (dfc.hdte_resid >= 0.9)], x='strike', y='leverage', color='hdte_resid', height=800)

In [ ]:
px.scatter(dfc[dfc.symbol.str.contains('QQQ|SPY') & (dfc.dte >= 90) & (dfc.dte <= 300) & (dfc.moneyness <= 1) & (dfc.hdte_resid >= 0.9) & (dfc.hdte_resid <= 0.99)], x='hdte_resid', y='leverage', color='expDt', height=800)

### The End